# Lesson 14: Message Trimming & Summarization

## Objective
Prevent context window overflow by **trimming** old messages and **summarizing** the conversation history before it grows too large.

## Limitation of Lesson 13
- ❌ Messages accumulated forever — context window overflow after many turns
- ❌ Expensive: sending 1000 messages to each LLM call
- ❌ Old irrelevant context may confuse the LLM

## What is NEW in Lesson 14?
- ✅ `trim_messages()` utility — keep only the N most recent messages
- ✅ **Summarization node** — LLM summarizes old messages into one compact summary
- ✅ **Conditional trimming** — only summarize when messages exceed a threshold
- ✅ The "sliding window + summary" memory pattern

## Theory: Two Strategies

### Strategy 1: Hard Trim (Sliding Window)
```
[msg1, msg2, msg3, msg4, msg5, msg6]
                 ↓  keep last 4
         [msg3, msg4, msg5, msg6]
```
Simple but loses ALL old context.

### Strategy 2: Summarize + Trim (Production Pattern)
```
[msg1, msg2, msg3, msg4, msg5, msg6]
                 ↓
[SystemMessage("Summary: user asked about 5+3=8 and 4×7=28..."),
 msg5, msg6]
```
Preserves important context in compressed form.

### When to use which
| Use Case | Strategy |
|----------|---------|
| Simple chatbots | Hard trim (last N messages) |
| Production agents | Summarize + trim |
| Long research sessions | Hierarchical summarization |


In [ ]:
# ── Cell 1: Imports + LLM Setup ──────────────────────────────────────────────
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

import os
import vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, trim_messages
from langchain_core.messages import RemoveMessage

from langgraph.graph import StateGraph, MessagesState, START, END
from typing import TypedDict, Optional, Annotated, Literal
from langgraph.graph.message import add_messages
from IPython.display import Image, display

load_dotenv()
vertexai.init(project=os.getenv("PROJECT_ID"), location=os.getenv("LOCATION"))
llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)

MAX_MESSAGES = 6    # Summarize when we exceed this many messages
KEEP_RECENT  = 4    # Keep this many recent messages after summarization

print(f"✓ Setup complete | MAX_MESSAGES={MAX_MESSAGES} | KEEP_RECENT={KEEP_RECENT}")


In [ ]:
# ── Cell 2: State with Summary Field ─────────────────────────────────────────
class ArithmeticState(MessagesState):
    last_result: Optional[float]
    summary: str     # Compressed summary of older messages

print("✓ State defined with summary field")


In [ ]:
# ── Cell 3: Summarization Node ────────────────────────────────────────────────
# This node runs BEFORE the main agent when messages are too long
# It summarizes old messages and removes them, keeping only the summary

def summarize_node(state: ArithmeticState) -> dict:
    messages = state["messages"]
    existing_summary = state.get("summary", "")
    
    print(f"  [summarize] Too many messages ({len(messages)}), summarizing...")
    
    # Build summarization prompt
    summary_prompt = "Summarize this arithmetic conversation in 2-3 sentences. " \
                     "Include all calculated results and operations performed."
    
    if existing_summary:
        summary_prompt = f"Previous summary: {existing_summary}\n\n" \
                        f"Extend the summary to include these additional messages:"
    
    msgs_to_summarize = messages[:-KEEP_RECENT]  # All but the last KEEP_RECENT
    msgs_text = "\n".join([
        f"{'User' if isinstance(m, HumanMessage) else 'Agent'}: {m.content}"
        for m in msgs_to_summarize
    ])
    
    response = llm.invoke([
        HumanMessage(content=f"{summary_prompt}\n\nConversation:\n{msgs_text}")
    ])
    new_summary = response.content.strip()
    print(f"  [summarize] Summary: {new_summary[:100]}...")
    
    # Remove the old messages (keep last KEEP_RECENT)
    # RemoveMessage tells the add_messages reducer to DELETE specific messages
    messages_to_remove = [RemoveMessage(id=m.id) for m in msgs_to_summarize]
    
    return {
        "messages": messages_to_remove,   # These get REMOVED by add_messages
        "summary": new_summary
    }

print("✓ Summarization node defined")
print("  Uses RemoveMessage to delete old messages from state")


In [ ]:
# ── Cell 4: Main Arithmetic Agent ────────────────────────────────────────────
SYSTEM_PROMPT = """You are an arithmetic assistant. Compute the answer and end with RESULT: <number>."""

def arithmetic_agent(state: ArithmeticState) -> dict:
    messages = state["messages"]
    summary  = state.get("summary", "")
    last_result = state.get("last_result")
    
    # Build context: system + optional summary + recent messages
    context = [SystemMessage(content=SYSTEM_PROMPT)]
    
    if summary:
        context.append(SystemMessage(
            content=f"Session summary (older context): {summary}"
        ))
    if last_result is not None:
        context.append(SystemMessage(content=f"Last result: {last_result}"))
    
    context.extend(messages)
    
    print(f"  [agent] {len(messages)} messages + summary='{summary[:40] if summary else 'none'}'")
    
    response = llm.invoke(context)
    ai_content = response.content.strip()
    print(f"  [agent] {ai_content[:80]}")
    
    result = last_result
    for line in ai_content.split("\n"):
        if "RESULT:" in line:
            try:
                result = float(line.split("RESULT:")[1].strip())
            except:
                pass
    
    return {
        "messages": [AIMessage(content=ai_content)],
        "last_result": result
    }

print("✓ Arithmetic agent defined")


In [ ]:
# ── Cell 5: Should-Summarize Router ──────────────────────────────────────────
def should_summarize(state: ArithmeticState) -> Literal["summarize", "agent"]:
    msg_count = len(state["messages"])
    if msg_count > MAX_MESSAGES:
        print(f"  [router] {msg_count} messages > {MAX_MESSAGES} → summarize first")
        return "summarize"
    print(f"  [router] {msg_count} messages ≤ {MAX_MESSAGES} → proceed to agent")
    return "agent"

print("✓ Should-summarize router defined")


In [ ]:
# ── Cell 6: Build Graph ──────────────────────────────────────────────────────
builder = StateGraph(ArithmeticState)

builder.add_node("summarize", summarize_node)
builder.add_node("agent",     arithmetic_agent)

# On entry: check if we need to summarize first
builder.add_conditional_edges(START, should_summarize, {
    "summarize": "summarize",
    "agent":     "agent"
})

# After summarizing, always proceed to agent
builder.add_edge("summarize", "agent")
builder.add_edge("agent",     END)

graph = builder.compile()
print("✓ Graph with summarization compiled")


In [ ]:
# ── Cell 7: Visualize ───────────────────────────────────────────────────────
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# ── Cell 8: Simulate Long Session (trigger summarization) ────────────────────
session_state = {"messages": [], "last_result": None, "summary": ""}

def ask(question: str):
    global session_state
    print(f"\n{'─'*55}")
    print(f"User: {question}  [messages in state: {len(session_state['messages'])}]")
    session_state["messages"] = session_state["messages"] + [HumanMessage(content=question)]
    out = graph.invoke(session_state)
    session_state = out
    ai_msg = out["messages"][-1].content
    print(f"Agent: {ai_msg[:100]}")
    print(f"  [msgs={len(out['messages'])}, result={out['last_result']}, has_summary={bool(out['summary'])}]")

# Generate enough turns to trigger summarization
questions = [
    "what is 5 plus 3?",
    "multiply that by 2",
    "what is 100 divided by 5?",
    "add 7 to that result",
    "what is 4 times 9?",
    "subtract 10 from the last result",
    "what is 15 plus 15?",         # This turn triggers summarization
    "divide that by 6",
]

for q in questions:
    ask(q)


In [ ]:
# ── Cell 9: Inspect Final State ───────────────────────────────────────────────
print("\n── Final State ──────────────────────────────────────")
print(f"  Messages remaining: {len(session_state['messages'])}")
print(f"  Last result: {session_state['last_result']}")
print(f"\n  Summary:")
print(f"  {session_state['summary']}")
print(f"\n  Remaining messages:")
for m in session_state["messages"]:
    role = "User" if isinstance(m, HumanMessage) else "Agent"
    print(f"    [{role}] {m.content[:60]}")


## Strategy Comparison

### Hard Trim (no summarization)
```python
from langchain_core.messages import trim_messages
trimmed = trim_messages(messages, max_tokens=1000, token_counter=len)
```
Fastest, but loses all old context.

### Summarize + Trim (production pattern, this lesson)
```
Before summarization:
  [Human, AI, Human, AI, Human, AI, Human, AI]  ← 8 messages

After summarization:
  [SystemMessage("Summary: user asked 5+3=8, then ×2=16...")  ← compressed
   Human, AI, Human, AI]                                       ← recent 4 kept
```

### `RemoveMessage` Pattern
```python
from langchain_core.messages import RemoveMessage
return {"messages": [RemoveMessage(id=msg.id) for msg in old_msgs]}
# The add_messages reducer DELETES messages whose IDs are in RemoveMessage objects
```

## Summary

| Pattern | Lesson 13 | Lesson 14 |
|---------|-----------|-----------|
| Memory size | Grows forever | Bounded (trim + summarize) |
| Old context | Always full | Compressed to summary |
| Production ready | No | Yes |

## Limitations → What Lesson 15 Solves
- ❌ Memory is still session-only — lost when the process restarts
- ❌ Can't recall calculations from last week's session
- ❌ **Lesson 15**: Long-term memory — persist past calculations across sessions
